In [2]:
pip install -qU langgraph


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [56]:
from typing import List, Dict
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, AIMessage
from langchain_ollama import ChatOllama
from dotenv import load_dotenv


In [57]:

load_dotenv()

class State(TypedDict):
    messages: List[Dict[str, str]]

graph_builder = StateGraph(State)

llm = ChatOllama(
    model="phi3:latest",
    temperature=0.5
)


In [59]:

def chatbot(state: State):
    formatted_messages = []
    for msg in state["messages"]:
        if msg["role"] == "user":
            formatted_messages.append(HumanMessage(content=msg["content"]))
        elif msg["role"] in ["assistant", "assistent"]:
            formatted_messages.append(AIMessage(content=msg["content"]))
            
    response = llm.invoke(formatted_messages)
    return {"messages": [{"role": "assistant", "content": response.content}]}


In [61]:

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)
graph = graph_builder.compile()

def stream_graph_updates(user_input: str):
    state = {"messages": [{"role": "user", "content": user_input}]}
    for event in graph.stream(state):
        for value in event.values():
            for msg in value["messages"]:
                if msg["role"] == "assistant":
                    print("Assistant:", msg["content"])




Adding a node to a graph that has already been compiled. This will not be reflected in the compiled graph.


ValueError: Node `chatbot` already present.

In [ ]:
print("Chatbot is ready! Type 'quit' to exit.\n")

while True:
    try:
        user_input = input("User: ")
        if user_input.lower() in ["quit", "exit", "q"]:
            print("Goodbye!")
            break
        stream_graph_updates(user_input)
    except Exception as e:
        print(f"Error: {e}")
        break

Chatbot is ready! Type 'quit' to exit.

